# 03 - Phan tich hieu qua noi dung

Notebook xep hang video/post theo reach, engagement, engagement rate va so sanh brand voi competitor flag trong export.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

posts = read_csv('data/processed/posts.csv')
content = read_csv('dashboard/exports/vw_content_performance.csv')
viral = read_csv('dashboard/exports/vw_viral_posts.csv')


In [ ]:
content_summary = content.sort_values(['is_competitor', 'total_reach'], ascending=[True, False])
display(content_summary.round(4))


In [ ]:
top_posts = viral.sort_values(['engagement_count', 'reach'], ascending=False).head(10)
display(top_posts[['page_name', 'full_date', 'reach', 'engagement_count', 'engagement_rate', 'caption']])


In [ ]:
brand = 'Highlands Coffee Vietnam'
brand_posts = posts[posts['page_name'].eq(brand)].copy()
brand_posts['engagement_count'] = brand_posts['likes'] + brand_posts['comments_count'] + brand_posts['shares']
brand_rank = brand_posts.sort_values(['engagement_rate', 'engagement_count'], ascending=False)
display(brand_rank[['content_text', 'date', 'reach', 'engagement_count', 'engagement_rate']].head(10))


In [ ]:
performance_bands = posts.assign(
    engagement_count=posts['likes'] + posts['comments_count'] + posts['shares'],
    performance_band=pd.qcut(posts['engagement_rate'].rank(method='first'), q=4, labels=['low', 'mid_low', 'mid_high', 'high'])
).groupby('performance_band', observed=True).agg(
    post_count=('post_id', 'count'),
    median_reach=('reach', 'median'),
    avg_engagement_rate=('engagement_rate', 'mean'),
    total_engagement=('engagement_count', 'sum'),
).reset_index()
display(performance_bands.round(4))


## Insight

- Key finding: Video la toan bo format trong export, nhung hieu qua tap trung vao mot nhom nho noi dung co reach lon.
- Supporting data: nhom non-competitor co 40 video, 6.22M reach va 1.10K engagements; nhom competitor flag co 25 video, 12.57K reach va 50 engagements.
- Recommended action: Tach rieng phan tich owned brand va noi dung nganh/creator de tranh video ngoai thuong hieu lam lech benchmark.